# Demo F: vLLM Engine Benchmarks

**Workshop Part 3** | LLM Inference at Scale | AI Engineering World's Fair 2026

**Platform:** Lightning.ai Studio (A100 GPU)

**Goal:** Prove that a production engine (vLLM) with optimizations enabled gives 10-20x improvement
over naive HuggingFace inference. Same model, same GPU, different flags.

## What we benchmark:
| Part | Optimization | vLLM Flag | Expected Gain |
|------|-------------|-----------|---------------|
| 1 | Baseline | (HuggingFace, no engine) | 1x |
| 2 | PagedAttention + Continuous Batching | (vLLM default) | 5-10x throughput |
| 3 | Prefix Caching | `--enable-prefix-caching` | 5-15x TTFT |
| 4 | KV Quantization | `--kv-cache-dtype fp8` | 2x capacity |
| 5 | Speculative Decoding | `--speculative-model TinyLlama` | 2-3x decode |

## Metrics we measure:
- **TTFT** (Time to First Token): latency before user sees anything
- **Throughput** (tok/s): total tokens generated per second across all users
- **Capacity**: max concurrent users before OOM

## Setup on Lightning.ai:
1. Create a Studio with A100 GPU
2. Install: `pip install vllm`
3. Download model: `hf download mistralai/Mistral-7B-v0.1`
4. Run this notebook top to bottom, starting/stopping server between sections


In [ ]:
# Demo F: vLLM Engine Benchmarks (Lightning.ai A100)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'openai', 'torch', 'transformers', 'accelerate',
                       'matplotlib', 'requests', 'tqdm', 'numpy<2', 'scipy>=1.14'])

import torch, time, requests
import matplotlib.pyplot as plt
from tqdm import tqdm
from openai import OpenAI
import concurrent.futures

# ─── Config ───
MODEL = 'mistralai/Mistral-7B-v0.1'
PORT = 8000
VLLM_URL = f'http://localhost:{PORT}'
BASE_URL = f'{VLLM_URL}/v1'

# OpenAI-compatible client (works with vLLM + SGLang)
client = OpenAI(base_url=BASE_URL, api_key='unused')

# ─── Benchmark Utility ───
def benchmark(prompts, max_tokens=50, label=''):
    """Send prompts concurrently, measure TTFT and throughput."""
    ttfts = []  # per-request TTFT in ms
    
    def single_request(prompt):
        t0 = time.perf_counter()
        first_token_time = None
        tokens = 0
        # Use streaming to measure TTFT
        stream = client.completions.create(
            model=MODEL, prompt=prompt, max_tokens=max_tokens,
            temperature=0, stream=True
        )
        for chunk in stream:
            if first_token_time is None:
                first_token_time = time.perf_counter()
            tokens += 1
        elapsed = time.perf_counter() - t0
        ttft = (first_token_time - t0) * 1000 if first_token_time else 0
        return {'ttft_ms': ttft, 'tokens': tokens, 'elapsed_s': elapsed}
    
    # Send all prompts concurrently
    t_total = time.perf_counter()
    with concurrent.futures.ThreadPoolExecutor(max_workers=len(prompts)) as pool:
        futures = [pool.submit(single_request, p) for p in prompts]
        request_results = [f.result() for f in tqdm(concurrent.futures.as_completed(futures),
                                                      total=len(prompts), desc=label)]
    wall_time = time.perf_counter() - t_total
    
    # Aggregate metrics
    total_tokens = sum(r['tokens'] for r in request_results)
    avg_ttft = sum(r['ttft_ms'] for r in request_results) / len(request_results)
    throughput = total_tokens / wall_time
    
    print(f'  [{label}] {len(prompts)} requests | {throughput:.0f} tok/s | avg TTFT: {avg_ttft:.0f} ms | wall: {wall_time:.1f}s')
    return {'throughput': throughput, 'ttft_ms': avg_ttft, 'wall_s': wall_time, 'total_tokens': total_tokens}

# ─── Metrics Helper ───
def get_kv_usage():
    """Get KV cache utilization from vLLM /metrics endpoint."""
    import re
    resp = requests.get(f'{VLLM_URL}/metrics')
    match = re.search(r'vllm:gpu_cache_usage_perc\s+([\d.]+)', resp.text)
    return float(match.group(1)) * 100 if match else 0.0

# Store results across experiments
results = {}

print('Setup complete.')
print(f'  Model: {MODEL}')
print(f'  Server: {VLLM_URL}')
print(f'  Client: OpenAI SDK (streaming for TTFT)')


In [ ]:
# --- Test Configuration ---
N_REQUESTS = 10  # number of concurrent requests per benchmark
N_TOKENS = 50  # max tokens to generate per request

# Standard test prompts (diverse topics to avoid caching bias)
PROMPTS = [
    "Explain the concept of attention mechanisms in transformers",
    "What are the key differences between CPU and GPU architectures",
    "Describe how PagedAttention works in vLLM",
    "What is the roofline model for GPU performance analysis",
    "Explain continuous batching in LLM serving systems",
    "How does speculative decoding accelerate inference",
    "What are the tradeoffs of KV cache quantization",
    "Describe the prefill vs decode phases of LLM inference",
    "How do mixture of experts models reduce compute costs",
    "What is tensor parallelism and when should you use it",
]

# Prefix prompts: shared system context + varying questions (tests prefix caching)
SYSTEM_PREFIX = (
    "You are an expert systems architect specializing in distributed ML infrastructure. "
    "You have deep knowledge of GPU memory hierarchies, CUDA programming, network topologies, "
    "and large-scale model serving. Answer questions precisely and technically. "
    "Focus on practical production considerations over theoretical ideals. "
    "Always mention relevant tradeoffs and failure modes."
)  # ~100 tokens shared prefix

PREFIX_PROMPTS = [
    f"{SYSTEM_PREFIX} How should I handle GPU OOM errors in production?",
    f"{SYSTEM_PREFIX} What monitoring metrics matter for LLM serving?",
    f"{SYSTEM_PREFIX} Compare disaggregated prefill vs colocated serving.",
    f"{SYSTEM_PREFIX} How does request scheduling affect tail latency?",
    f"{SYSTEM_PREFIX} What causes throughput degradation under high concurrency?",
    f"{SYSTEM_PREFIX} Explain memory fragmentation in KV cache management.",
    f"{SYSTEM_PREFIX} How to right-size GPU instances for a 7B model?",
    f"{SYSTEM_PREFIX} What is the impact of sequence length on batch size?",
    f"{SYSTEM_PREFIX} Compare chunked prefill vs full prefill strategies.",
    f"{SYSTEM_PREFIX} How does FlashAttention reduce memory bandwidth pressure?",
]

# Results dictionary: stores metrics for each configuration
results = {}
print(f"Config: {N_REQUESTS} requests, {N_TOKENS} tokens each")
print(f"Prefix prompts share ~100 token system context")

In [ ]:
# --- HuggingFace Baseline (Sequential) ---
# Load model on GPU for fair comparison
print("Loading Mistral-7B via HuggingFace...")
hf_tokenizer = AutoTokenizer.from_pretrained(MODEL)
hf_tokenizer.pad_token_id = hf_tokenizer.eos_token_id  # required for open-ended generation
hf_model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="auto")

# Warmup: 3 calls with SEPARATE warmup prompt to compile CUDA kernels
print("Warming up (3 calls)...")
for _ in range(3):
    warmup_ids = hf_tokenizer(WARMUP_PROMPT, return_tensors="pt").input_ids.to("cuda")
    _ = hf_model.generate(warmup_ids, max_new_tokens=10, pad_token_id=hf_tokenizer.eos_token_id)

# Benchmark: sequential generation (HF can't serve concurrent requests)
hf_ttfts = []  # time to first token per request
hf_total_tokens = 0
hf_start = time.perf_counter()

for prompt in tqdm(PROMPTS, desc="HF baseline"):
    input_ids = hf_tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
    t0 = time.perf_counter()
    # Generate and measure timing
    with torch.no_grad():
        output = hf_model.generate(input_ids, max_new_tokens=N_TOKENS, pad_token_id=hf_tokenizer.eos_token_id)
    t1 = time.perf_counter()
    generated_tokens = output.shape[1] - input_ids.shape[1]  # new tokens only
    hf_ttfts.append(t1 - t0)  # HF TTFT ~ total time (no streaming)
    hf_total_tokens += generated_tokens

hf_wall = time.perf_counter() - hf_start  # total wall time
hf_throughput = hf_total_tokens / hf_wall  # tokens/sec

# Store HF results
results["HF Baseline"] = {"avg_ttft": np.mean(hf_ttfts), "p50_ttft": np.median(hf_ttfts),
                           "throughput_tps": hf_throughput, "wall_time": hf_wall, "total_tokens": hf_total_tokens}
print(f"HF: {hf_throughput:.1f} tok/s, avg latency {np.mean(hf_ttfts)*1000:.0f}ms")

# Free GPU memory for vLLM server
del hf_model, hf_tokenizer
torch.cuda.empty_cache()
print("GPU freed for vLLM server.")

In [ ]:
# --- Running Comparison (all experiments so far) ---
collected_engines = list(results.keys())
collected_tp = [results[eng]['throughput'] for eng in collected_engines]

fig_cmp, ax_cmp = plt.subplots(figsize=(max(6, len(collected_engines)*2), 4))
cmp_colors = ['#ffe4e6'] + ['#dcfce7'] * (len(collected_engines) - 1)
ax_cmp.bar(collected_engines, collected_tp, color=cmp_colors, edgecolor='#000', linewidth=1.2)
for ci, (eng, tp) in enumerate(zip(collected_engines, collected_tp)):
    ax_cmp.text(ci, tp + max(collected_tp)*0.02, f'{tp:.0f}', ha='center', fontsize=10, fontweight='bold')
    if ci > 0:
        speedup = tp / collected_tp[0]
        ax_cmp.text(ci, tp*0.5, f'{speedup:.1f}x', ha='center', fontsize=12, color='#166534', fontweight='bold')
ax_cmp.set_ylabel('Throughput (tok/s)')
ax_cmp.set_title('Running Comparison (vs HF Baseline)', fontweight='bold')
ax_cmp.spines['top'].set_visible(False)
ax_cmp.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


## Experiment 0: Observing KV Cache in Action

Before benchmarking optimizations, let's SEE the KV cache growing in real time.
vLLM exposes a `/metrics` endpoint (Prometheus format) that shows:
- KV cache utilization (%)
- Number of running/waiting requests
- GPU memory allocated

We send increasing concurrent requests and watch the KV cache fill up.

**Start vLLM server first** (same command as Experiment 1):
```bash
python -m vllm.entrypoints.openai.api_server \
    --model mistralai/Mistral-7B-v0.1 \
    --dtype float16 \
    --gpu-memory-utilization 0.90 \
    --port 8000
```


In [ ]:
# --- Experiment 0: KV Cache Visualization ---
import re

def get_kv_cache_usage():
    """Query vLLM /metrics endpoint for KV cache utilization."""
    resp = requests.get(f'{BASE_URL.replace("/v1", "")}/metrics')
    text = resp.text
    # Parse Prometheus metrics
    usage = re.search(r'vllm:gpu_cache_usage_perc\s+([\d.]+)', text)
    return float(usage.group(1)) * 100 if usage else 0.0

# Send increasing concurrent requests and track KV cache usage
import concurrent.futures

kv_usage_over_load = []  # (n_concurrent, kv_usage_pct)
load_levels = [1, 5, 10, 20, 30]

print('Watching KV cache fill as we increase concurrent users...')
for n_conc in tqdm(load_levels, desc='Load levels'):
    # Send n_conc requests concurrently (long output to keep KV alive)
    conc_prompts = [f'Write a detailed essay about topic {i}:' for i in range(n_conc)]
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=n_conc) as executor:
        futures = [executor.submit(
            requests.post, f'{BASE_URL}/completions',
            json={'model': MODEL, 'prompt': p, 'max_tokens': 100, 'temperature': 0}
        ) for p in conc_prompts]
        
        # While requests are in-flight, sample KV cache usage
        import time as time_mod
        time_mod.sleep(0.5)  # Let requests start
        kv_pct = get_kv_cache_usage()
        kv_usage_over_load.append((n_conc, kv_pct))
        print(f'  {n_conc:>2} concurrent users -> KV cache: {kv_pct:.1f}% full')
        
        # Wait for all to finish
        concurrent.futures.wait(futures)

# Plot KV cache usage vs load
fig_kv, ax_kv = plt.subplots(figsize=(8, 4))
kv_loads = [x[0] for x in kv_usage_over_load]
kv_pcts = [x[1] for x in kv_usage_over_load]

ax_kv.plot(kv_loads, kv_pcts, 'o-', color='#991b1b', linewidth=2, markersize=8)
ax_kv.axhline(y=90, color='#64748b', linestyle='--', label='90% utilization limit')
ax_kv.set_xlabel('Concurrent Users', fontsize=11)
ax_kv.set_ylabel('KV Cache Usage (%)', fontsize=11)
ax_kv.set_title('KV Cache Fills Up With Concurrent Users', fontsize=12, fontweight='bold')
ax_kv.set_ylim(0, 100)
ax_kv.legend()
ax_kv.spines['top'].set_visible(False)
ax_kv.spines['right'].set_visible(False)
for xi, yi in zip(kv_loads, kv_pcts):
    ax_kv.annotate(f'{yi:.0f}%', (xi, yi), textcoords='offset points', xytext=(0, 10), ha='center')
plt.tight_layout()
plt.show()

print(f'\nThe KV cache is the bottleneck. More users = more KV = closer to OOM.')
print(f'Every optimization after this reduces KV pressure or improves throughput.')


## Experiment 1: PagedAttention + Continuous Batching (vLLM)

PagedAttention is vLLM's core innovation: virtual memory for KV cache.
Instead of pre-allocating contiguous blocks (50-70% wasted), it uses a page table.

We benchmark throughput with many concurrent users to show the capacity gain.

### Start vLLM server (run in terminal or background):

Open a **terminal** in Lightning.ai Studio and run:

```bash
pip install vllm
python -m vllm.entrypoints.openai.api_server \
    --model mistralai/Mistral-7B-v0.1 \
    --dtype float16 \
    --port 8000
```

Wait until you see `Uvicorn running on http://0.0.0.0:8000`. Then run the next cell.

In [ ]:
# --- vLLM Default Benchmark ---
# Verify server is running
try:
    health = requests.get(f"{BASE_URL}/health", timeout=5)
    assert health.status_code == 200, f"Server returned {health.status_code}"
    print("vLLM server healthy ✓")
except Exception as e:
    raise RuntimeError(f"vLLM server not reachable: {e}. Start it in a terminal first.")

# Warmup with separate prompt (compiles CUDA graphs)
print("Warming up vLLM...")
_ = requests.post(f"{BASE_URL}/v1/completions",
                  json={"model": MODEL, "prompt": WARMUP_PROMPT, "max_tokens": 10})

# Run concurrent benchmark
vllm_results = benchmark_concurrent(PROMPTS, max_tokens=N_TOKENS, n_workers=N_REQUESTS, label="vLLM default")
results["vLLM"] = vllm_results
print(f"vLLM: {vllm_results['throughput_tps']:.1f} tok/s, avg TTFT {vllm_results['avg_ttft']*1000:.0f}ms")
print(f"Speedup over HF: {vllm_results['throughput_tps']/results['HF Baseline']['throughput_tps']:.1f}x")

In [ ]:
# --- Running Comparison (all experiments so far) ---
collected_engines = list(results.keys())
collected_tp = [results[eng]['throughput'] for eng in collected_engines]

fig_cmp, ax_cmp = plt.subplots(figsize=(max(6, len(collected_engines)*2), 4))
cmp_colors = ['#ffe4e6'] + ['#dcfce7'] * (len(collected_engines) - 1)
ax_cmp.bar(collected_engines, collected_tp, color=cmp_colors, edgecolor='#000', linewidth=1.2)
for ci, (eng, tp) in enumerate(zip(collected_engines, collected_tp)):
    ax_cmp.text(ci, tp + max(collected_tp)*0.02, f'{tp:.0f}', ha='center', fontsize=10, fontweight='bold')
    if ci > 0:
        speedup = tp / collected_tp[0]
        ax_cmp.text(ci, tp*0.5, f'{speedup:.1f}x', ha='center', fontsize=12, color='#166534', fontweight='bold')
ax_cmp.set_ylabel('Throughput (tok/s)')
ax_cmp.set_title('Running Comparison (vs HF Baseline)', fontweight='bold')
ax_cmp.spines['top'].set_visible(False)
ax_cmp.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


## Experiment 2: Prefix Caching

All 50 requests above share the same 400-token system prompt.
With prefix caching enabled, vLLM computes that prefix ONCE and reuses it.

Stop the server (Ctrl+C) and restart with:

```bash
python -m vllm.entrypoints.openai.api_server \
    --model mistralai/Mistral-7B-v0.1 \
    --dtype float16 \
    --port 8000 \
    --enable-prefix-caching
```

Prefix caching reuses KV cache blocks for shared prompt prefixes, reducing TTFT for requests that share context.

In [ ]:
# --- Prefix Caching Benchmark ---
# Verify server restarted
health = requests.get(f"{BASE_URL}/health", timeout=5)
assert health.status_code == 200, "Server not ready"

# Warmup
_ = requests.post(f"{BASE_URL}/v1/completions",
                  json={"model": MODEL, "prompt": WARMUP_PROMPT, "max_tokens": 10})

# Batch 1: COLD (prefix not yet cached)
print("Batch 1: Cold prefix (first time seeing system prompt)...")
cold_results = benchmark_concurrent(PREFIX_PROMPTS[:5], max_tokens=N_TOKENS, n_workers=5, label="cold prefix")

# Batch 2: WARM (prefix should be cached from batch 1)
print("Batch 2: Warm prefix (system prompt cached)...")
warm_results = benchmark_concurrent(PREFIX_PROMPTS[5:], max_tokens=N_TOKENS, n_workers=5, label="warm prefix")

# Store combined results (warm is the real benefit)
results["vLLM+Prefix"] = warm_results
prefix_speedup = cold_results["avg_ttft"] / warm_results["avg_ttft"]  # TTFT improvement
print(f"Cold TTFT: {cold_results['avg_ttft']*1000:.0f}ms -> Warm TTFT: {warm_results['avg_ttft']*1000:.0f}ms")
print(f"Prefix caching TTFT speedup: {prefix_speedup:.2f}x")

# Show cache stats
cache_stats = get_vllm_cache_stats()
print("\nCache metrics:")
for line in cache_stats:
    print(f"  {line}")

In [ ]:
# --- Running Comparison (all experiments so far) ---
collected_engines = list(results.keys())
collected_tp = [results[eng]['throughput'] for eng in collected_engines]

fig_cmp, ax_cmp = plt.subplots(figsize=(max(6, len(collected_engines)*2), 4))
cmp_colors = ['#ffe4e6'] + ['#dcfce7'] * (len(collected_engines) - 1)
ax_cmp.bar(collected_engines, collected_tp, color=cmp_colors, edgecolor='#000', linewidth=1.2)
for ci, (eng, tp) in enumerate(zip(collected_engines, collected_tp)):
    ax_cmp.text(ci, tp + max(collected_tp)*0.02, f'{tp:.0f}', ha='center', fontsize=10, fontweight='bold')
    if ci > 0:
        speedup = tp / collected_tp[0]
        ax_cmp.text(ci, tp*0.5, f'{speedup:.1f}x', ha='center', fontsize=12, color='#166534', fontweight='bold')
ax_cmp.set_ylabel('Throughput (tok/s)')
ax_cmp.set_title('Running Comparison (vs HF Baseline)', fontweight='bold')
ax_cmp.spines['top'].set_visible(False)
ax_cmp.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


## Experiment 3: KV Cache Quantization

Stop the server and restart with FP8 KV cache:

```bash
python -m vllm.entrypoints.openai.api_server \
    --model mistralai/Mistral-7B-v0.1 \
    --dtype float16 \
    --port 8000 \
    --kv-cache-dtype fp8
```

FP8 KV cache halves memory per token, allowing ~2x more concurrent requests before OOM.

In [ ]:
# --- KV Cache Quantization Benchmark (High Concurrency Stress Test) ---
health = requests.get(f"{BASE_URL}/health", timeout=5)
assert health.status_code == 200, "Server not ready"

# Warmup
_ = requests.post(f"{BASE_URL}/v1/completions",
                  json={"model": MODEL, "prompt": WARMUP_PROMPT, "max_tokens": 10})

# Stress test: 20 concurrent requests (tests capacity gain from quantized KV)
stress_prompts = PROMPTS * 2  # 20 prompts total
kvq_results = benchmark_concurrent(stress_prompts, max_tokens=N_TOKENS, n_workers=20, label="KV quant 20x")
results["vLLM+KVQuant"] = kvq_results
print(f"KV Quant (20 concurrent): {kvq_results['throughput_tps']:.1f} tok/s, avg TTFT {kvq_results['avg_ttft']*1000:.0f}ms")
print(f"Sustained {kvq_results['total_tokens']} tokens across 20 concurrent requests without OOM")

In [ ]:
# --- Running Comparison (all experiments so far) ---
collected_engines = list(results.keys())
collected_tp = [results[eng]['throughput'] for eng in collected_engines]

fig_cmp, ax_cmp = plt.subplots(figsize=(max(6, len(collected_engines)*2), 4))
cmp_colors = ['#ffe4e6'] + ['#dcfce7'] * (len(collected_engines) - 1)
ax_cmp.bar(collected_engines, collected_tp, color=cmp_colors, edgecolor='#000', linewidth=1.2)
for ci, (eng, tp) in enumerate(zip(collected_engines, collected_tp)):
    ax_cmp.text(ci, tp + max(collected_tp)*0.02, f'{tp:.0f}', ha='center', fontsize=10, fontweight='bold')
    if ci > 0:
        speedup = tp / collected_tp[0]
        ax_cmp.text(ci, tp*0.5, f'{speedup:.1f}x', ha='center', fontsize=12, color='#166534', fontweight='bold')
ax_cmp.set_ylabel('Throughput (tok/s)')
ax_cmp.set_title('Running Comparison (vs HF Baseline)', fontweight='bold')
ax_cmp.spines['top'].set_visible(False)
ax_cmp.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


## Experiment 4: Speculative Decoding with vllm

Stop the server and restart with a draft model:

```bash
python -m vllm.entrypoints.openai.api_server \
    --model mistralai/Mistral-7B-v0.1 \
    --dtype float16 \
    --port 8000 \
    --speculative-model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
    --num-speculative-tokens 5
```

Speculative decoding uses a small draft model to propose tokens verified in parallel by the main model, accelerating decode-bound generation.

In [ ]:
# --- Speculative Decoding Benchmark ---
try:
    health = requests.get(f"{BASE_URL}/health", timeout=5)
    assert health.status_code == 200, "Server not ready"

    # Warmup
    _ = requests.post(f"{BASE_URL}/v1/completions",
                      json={"model": MODEL, "prompt": WARMUP_PROMPT, "max_tokens": 10})

    # Benchmark: speculative should improve decode throughput
    spec_results = benchmark_concurrent(PROMPTS, max_tokens=N_TOKENS, n_workers=N_REQUESTS, label="speculative")
    results["vLLM+Spec"] = spec_results
    spec_vs_default = spec_results["throughput_tps"] / results["vLLM"]["throughput_tps"]
    print(f"Speculative: {spec_results['throughput_tps']:.1f} tok/s, avg TTFT {spec_results['avg_ttft']*1000:.0f}ms")
    print(f"Decode speedup vs default vLLM: {spec_vs_default:.2f}x")

except Exception as e:
    # Speculative decoding may OOM on smaller GPUs
    print(f"Speculative decoding failed (likely OOM): {e}")
    print("This is expected if GPU VRAM is insufficient for target + draft model")
    results["vLLM+Spec"] = None

In [ ]:
# --- Running Comparison (all experiments so far) ---
collected_engines = list(results.keys())
collected_tp = [results[eng]['throughput'] for eng in collected_engines]

fig_cmp, ax_cmp = plt.subplots(figsize=(max(6, len(collected_engines)*2), 4))
cmp_colors = ['#ffe4e6'] + ['#dcfce7'] * (len(collected_engines) - 1)
ax_cmp.bar(collected_engines, collected_tp, color=cmp_colors, edgecolor='#000', linewidth=1.2)
for ci, (eng, tp) in enumerate(zip(collected_engines, collected_tp)):
    ax_cmp.text(ci, tp + max(collected_tp)*0.02, f'{tp:.0f}', ha='center', fontsize=10, fontweight='bold')
    if ci > 0:
        speedup = tp / collected_tp[0]
        ax_cmp.text(ci, tp*0.5, f'{speedup:.1f}x', ha='center', fontsize=12, color='#166534', fontweight='bold')
ax_cmp.set_ylabel('Throughput (tok/s)')
ax_cmp.set_title('Running Comparison (vs HF Baseline)', fontweight='bold')
ax_cmp.spines['top'].set_visible(False)
ax_cmp.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


## Final Comparison: All Optimizations

We ran the same test across 5 configurations. Same model. Same GPU. Same prompts.
Each flag stacks on top of the previous. The final chart shows the cumulative effect.

**What each optimization addressed:**
- **vLLM default:** PagedAttention (no fragmentation) + continuous batching (no padding) → throughput
- **Prefix caching:** Shared system prompts computed once → TTFT for repeat users
- **KV quantization:** FP8 KV cache → memory capacity (more users before OOM)
- **Speculative decoding:** Draft model verifies in parallel → decode speed (ITL)


In [ ]:
# --- Final Comparison Chart ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Filter out None results (failed benchmarks)
valid = {k: v for k, v in results.items() if v is not None}
names = list(valid.keys())
throughputs = [valid[k]["throughput_tps"] for k in names]
ttfts = [valid[k]["avg_ttft"] * 1000 for k in names]  # convert to ms

# Colors for each config
colors = ["#64748b", "#2563eb", "#059669", "#d97706", "#7c3aed"][:len(names)]

# Left plot: throughput bars
bars = ax1.bar(names, throughputs, color=colors, edgecolor="black", linewidth=0.8)
ax1.set_ylabel("Throughput (tokens/sec)", fontsize=11)
ax1.set_title("Throughput Comparison", fontsize=13, fontweight="bold")
ax1.set_xticklabels(names, rotation=15, ha="right", fontsize=9)
# Add value labels on bars
for bar, val in zip(bars, throughputs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f"{val:.0f}",
             ha="center", va="bottom", fontsize=9, fontweight="bold")

# Right plot: TTFT line + bars
ax2.bar(names, ttfts, color=colors, edgecolor="black", linewidth=0.8, alpha=0.7)
ax2.plot(names, ttfts, "ko-", markersize=8, linewidth=2)  # line overlay
ax2.set_ylabel("Avg TTFT (ms)", fontsize=11)
ax2.set_title("Time to First Token", fontsize=13, fontweight="bold")
ax2.set_xticklabels(names, rotation=15, ha="right", fontsize=9)

plt.tight_layout()
plt.savefig("demo_f_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# Print speedup table
hf_tps = results["HF Baseline"]["throughput_tps"]
print("\n" + "="*50)
print(f"{'Config':<20} {'tok/s':>8} {'TTFT ms':>8} {'vs HF':>8}")
print("-"*50)
for name, r in valid.items():
    speedup = r["throughput_tps"] / hf_tps
    print(f"{name:<20} {r['throughput_tps']:>8.1f} {r['avg_ttft']*1000:>8.0f} {speedup:>7.1f}x")
print("="*50)

## Summary: vLLM Optimization Flags

| Flag | What It Does | Best For |
|------|-------------|----------|
| (default) | PagedAttention, continuous batching | General serving baseline |
| `--enable-prefix-caching` | Reuses KV blocks for shared prefixes | Chat with system prompts, RAG |
| `--kv-cache-dtype fp8` | Quantizes KV cache to 8-bit | High concurrency, long contexts |
| `--speculative-model` | Draft model proposes, main verifies | Low-latency decode-bound tasks |

**Key Takeaways:**
- vLLM's PagedAttention alone gives 5-15x throughput over HF sequential
- Prefix caching reduces TTFT for repeated system prompts (chat/RAG workloads)
- FP8 KV cache doubles capacity at minimal quality loss
- Speculative decoding trades memory for decode speed (best with short outputs)

**Next:** Demo G explores SGLang's RadixAttention for comparison.